In [1]:
import os
import cv2
import pywt
import numpy as np
import matplotlib.pyplot as plt
import random

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    confusion_matrix
)

# 1. HÀM TẠO WAVELET HASH
def generate_wavelet_hash(img, hash_size=16):
    coeffs = pywt.wavedec2(img, 'db4', level=1)
    cA = coeffs[0]

    cA_resized = cv2.resize(cA, (hash_size, hash_size))
    median_val = np.median(cA_resized)

    hash_matrix = (cA_resized > median_val).astype(int)
    return hash_matrix


# 2. HÀM Hamming Distance
def hamming_dist(h1, h2):
    return np.count_nonzero(h1 != h2)


# 3. LOAD DATA
DATA_DIR = "Hinhanh"

data_list = []

for label in os.listdir(DATA_DIR):
    class_path = os.path.join(DATA_DIR, label)

    if not os.path.isdir(class_path):
        continue

    for file in os.listdir(class_path):
        img_path = os.path.join(class_path, file)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # Preprocessing
        img = cv2.resize(img, (128, 128))
        img = cv2.equalizeHist(img)

        hash_matrix = generate_wavelet_hash(img)

        data_list.append({
            "img": img,
            "label": label,
            "hash": hash_matrix,
            "name": file
        })

print("Tổng số ảnh:", len(data_list))


# 4. VISUALIZE (DEMO)
def visualize_pair(mode="SAME"):
    if mode == "SAME":
        labels = list(set([d['label'] for d in data_list]))
        valid_labels = [l for l in labels if sum(d['label']==l for d in data_list) >= 2]

        chosen_label = random.choice(valid_labels)
        idxs = [i for i, d in enumerate(data_list) if d['label'] == chosen_label]
        i, j = random.sample(idxs, 2)
        title = f"CÙNG LỚP ({chosen_label})"

    else:
        i = random.randint(0, len(data_list)-1)
        j = random.choice([k for k in range(len(data_list)) if data_list[k]['label'] != data_list[i]['label']])
        title = f"KHÁC LỚP ({data_list[i]['label']} vs {data_list[j]['label']})"

    d1 = data_list[i]
    d2 = data_list[j]

    dist = hamming_dist(d1['hash'], d2['hash'])

    plt.figure(figsize=(8,4))

    plt.subplot(1,2,1)
    plt.imshow(d1['img'], cmap='gray')
    plt.title(f"{d1['label']}")

    plt.subplot(1,2,2)
    plt.imshow(d2['img'], cmap='gray')
    plt.title(f"{d2['label']}\nDist={dist}")

    plt.suptitle(title)
    plt.show()


visualize_pair("SAME")
visualize_pair("DIFF")


# 5. TẠO DATA PAIRS
distances = []
y_true = []

for i in range(len(data_list)):
    for j in range(i+1, len(data_list)):
        d = hamming_dist(data_list[i]['hash'], data_list[j]['hash'])
        distances.append(d)

        if data_list[i]['label'] == data_list[j]['label']:
            y_true.append(1)
        else:
            y_true.append(0)


# 6. AUTO THRESHOLD (TỐI ƯU F1)
def find_best_threshold(distances, y_true):
    best_f1 = 0
    best_t = 0

    for t in range(5, 100):
        y_pred = [1 if d < t else 0 for d in distances]
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t


threshold = find_best_threshold(distances, y_true)
print("Ngưỡng tối ưu:", threshold)


# 7. PREDICTION
y_pred = [1 if d < threshold else 0 for d in distances]

# 8. EVALUATION
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp)

print("\n===== KẾT QUẢ =====")
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1-score: {f1:.4f}")

scores = [-d for d in distances]  # distance nhỏ = giống

fpr, tpr, _ = roc_curve(y_true, scores)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import os
import cv2
import pywt
import numpy as np
import matplotlib.pyplot as plt


def wavelet_hash(img, hash_size=64):
    coeffs = pywt.wavedec2(img, 'db4', level=1)
    cA = coeffs[0]

    cA = cv2.resize(cA, (hash_size, hash_size))
    median = np.median(cA)

    return (cA > median).astype(int)

def hamming(a, b):
    return np.sum(a != b)


def build_database(folder):
    db = []

    for label in os.listdir(folder):
        class_path = os.path.join(folder, label)

        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            path = os.path.join(class_path, file)

            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            img = cv2.resize(img, (128, 128))
            img = cv2.equalizeHist(img)

            h = wavelet_hash(img)

            db.append({
                "path": path,
                "label": label,
                "hash": h
            })

    print("✔ Database size:", len(db))
    return db

# 3. SEARCH
def search(query_path, db, top_k=3):
    print(" Query:", query_path)

    query = cv2.imread(query_path, cv2.IMREAD_GRAYSCALE)

    if query is None:
        raise ValueError(f"❌ Không đọc được ảnh: {query_path}")

    query = cv2.resize(query, (128, 128))
    query = cv2.equalizeHist(query)

    q_hash = wavelet_hash(query)

    results = []

    for item in db:
        d = hamming(q_hash, item["hash"])
        results.append((item["path"], item["label"], d))

    # sort theo distance
    results = sorted(results, key=lambda x: x[2])

    return results[:top_k]


def show_results(query_path, results):
    plt.figure(figsize=(12,4))

    # Query
    img = cv2.imread(query_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(1, len(results)+1, 1)
    plt.imshow(img)
    plt.title("Query")

    # Results
    for i, (path, label, dist) in enumerate(results):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(1, len(results)+1, i+2)
        plt.imshow(img)
        plt.title(f"{label}\nd={dist}")

    plt.tight_layout()
    plt.show()

# 5. RUN

DATASET_PATH = "Hinhanh"

db = build_database(DATASET_PATH)

query_path = r"Hinhanh\Meomeo\anh12.jpg"

results = search(query_path, db, top_k=3)

show_results(query_path, results)

ModuleNotFoundError: No module named 'pywt'